<a href="https://colab.research.google.com/github/hmontUSC/movie-recommender-svd/blob/main/notebooks/04_what_it_learned.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Milestone 4: What Did It Learn, and What Would It Recommend to You?

This notebook opens up the SVD to name its hidden factors and shows what it recommends to a new user from just ten ratings.

## 1. Set up

In [ ]:
USER, REPO = "hmontUSC", "movie-recommender-svd"
!rm -rf {REPO}
!git clone -q https://github.com/{USER}/{REPO}.git
import sys
sys.path.insert(0, f"{REPO}/src")
from evaluate import to_matrix
from models import baseline_matrix, rank_k

This code downloads a fresh copy of the repository into Colab and imports the functions this notebook needs from `src`.

In [ ]:
import os, urllib.request, zipfile
import numpy as np
import pandas as pd

if not os.path.exists("ml-100k"):
    urllib.request.urlretrieve(
        "https://files.grouplens.org/datasets/movielens/ml-100k.zip",
        "ml-100k.zip")
    zipfile.ZipFile("ml-100k.zip").extractall()

ratings = pd.read_csv("ml-100k/u.data", sep="\t",
                      names=["user", "item", "rating", "time"])


This code downloads MovieLens 100K if it is not already there and loads all 100,000 ratings into a table.

In [ ]:
titles = pd.read_csv("ml-100k/u.item", sep="|", encoding="latin-1",
                     header=None, usecols=[1])[1].to_numpy()
print(len(titles), titles[49])

This code reads the movie titles into an array, so movie number 50 is `titles[49]`. The `latin-1` encoding handles titles with accented letters.

## 2. Fit Model 2 on all the ratings

In [ ]:
R = to_matrix(ratings)
known = ~np.isnan(R)
B = baseline_matrix(R)
X = np.where(known, R - B, 0.0)
U, s, Vt = np.linalg.svd(X, full_matrices=False)

n_ratings = known.sum(axis=0)
popular = n_ratings >= 50
print("popular movies:", popular.sum())

This is Model 2 from Milestone 3, fitted on all the ratings instead of a training set, because here we are using the model rather than measuring it. `popular` marks movies with at least 50 ratings, since a movie rated by only a few people can land anywhere by chance.

## 3. What did it learn? Naming the hidden factors

In [ ]:
def show_factor(f, n=6):
    """Popular movies at the two ends of hidden factor f (f = 1, 2, ...)."""
    v = Vt[f - 1]
    idx = np.where(popular)[0]
    order = idx[np.argsort(v[idx])]
    print(f"Factor {f}, one end:")
    for m in order[:n]:
        print(f"  {v[m]:+.3f} {titles[m]}")
    print(f"Factor {f}, other end:")
    for m in order[::-1][:n]:
        print(f"  {v[m]:+.3f} {titles[m]}")

show_factor(1)

Each row of `Vt` gives every movie a score on one hidden factor. This function sorts the popular movies by that score and prints the six at each end, which are the movies the factor separates most. (Signs can be flipped by the SVD, so the two ends may swap, but the movies at each end stay the same.)

In [ ]:
show_factor(3)

The same function applied to factor 3.

## 4. What would it recommend to you?

In [ ]:
def find(word):
    """Print the ID and title of every movie whose title contains word."""
    for m, t in enumerate(titles):
        if word.lower() in t.lower():
            print(m + 1, t)

find("alien")

This function looks up movie IDs by searching titles, ignoring capitals. It prints `m + 1` because movie IDs start at 1 while Python positions start at 0.

In [ ]:
my_ratings = {50: 5, 172: 5, 183: 5, 195: 5, 89: 4, 1: 4,
              143: 1, 451: 1, 88: 1, 739: 2}

A dictionary pairing movie IDs with a rating: Star Wars, The Empire Strikes Back, Alien, The Terminator, Blade Runner and Toy Story rated high, and The Sound of Music, Grease, Sleepless in Seattle and Pretty Woman rated low.

In [ ]:
me = np.full((1, R.shape[1]), np.nan)
for movie, stars in my_ratings.items():
    me[0, movie - 1] = stars
R_me = np.vstack([R, me])
print(R_me.shape)

known_me = ~np.isnan(R_me)
B_me = baseline_matrix(R_me)
X_me = np.where(known_me, R_me - B_me, 0.0)
U, s, Vt = np.linalg.svd(X_me, full_matrices=False)
K = 10
P = rank_k(U, s, Vt, K) + B_me

This code builds a one-row matrix holding only my ratings, stacks it under `R` as user 944, and refits Model 2 with k = 10. My predictions are the last row, `P[-1]`.

In [ ]:
mine = P[-1]
base = B_me[-1]
can_pick = popular & np.isnan(R_me[-1])

def top(scores, n=10):
    """The n highest-scoring movies among those we may recommend."""
    return np.argsort(-np.where(can_pick, scores, -np.inf))[:n]

print("Model 2 recommends:")
for m in top(mine):
    print(f"  {mine[m]:.2f} {titles[m]}")
print("Baseline recommends:")
for m in top(base):
    print(f"  {base[m]:.2f} {titles[m]}")
print("in both lists:", len(set(top(mine)) & set(top(base))))

`can_pick` is True for popular movies I have not rated. `top` gives every other movie a score of minus infinity so it never makes the list, then sorts from highest to lowest. The last line counts how many movies appear in both top-10 lists.

In [ ]:
personal = mine - base
print("Moved up most for me:")
for m in top(personal, 6):
    print(f"  {personal[m]:+.2f} {titles[m]}")
print("Moved down most for me:")
for m in top(-personal, 6):
    print(f"  {personal[m]:+.2f} {titles[m]}")

The difference between the model's prediction and the baseline is exactly what the SVD added for me. This shows which movies it pushed up and down.

## What we found

Without being told any genres, the SVD discovered a "sci-fi and action vs. musicals and family films" factor (factor 3); from just 10 of my ratings it moved Star Trek: First Contact up by 0.22 stars and The English Patient down by 0.36, although 6 of its top 10 picks were the same crowd favorites the simple average recommends.